# Project 2

Author: Cameron Mullaney

This is the second project for ST554.

DESCRIPTION AND NARRATIVE

# Part I

Part I is split up into two major tasks :
    A. The creation of my "Proj2Script.py" file
    B. The use of the SparkDataCheck class and the methods created within. 
    
Piece A is outlined thoroughly in the script file, and piece B is below.
I will be shwocasing the methods created in the script to analyze and validate data from a csv of air quality data.

Loading in the tools I'll need, as well as my "Proj2Script.py" file from piece A

In [4]:
import pandas as pd
import numpy as np
import math
from pyspark.sql import SparkSession
from Proj2Script import *

Setting up my spark session, and reading in the data! 

In [7]:
spark = SparkSession.builder.appName('my_app').getOrCreate()

In [3]:
airdata = SparkDataCheck.fromcsv("air.csv", spark)
airdata.df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-23 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-23 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-23 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-23 21:00:00|   2.2|       137

26/03/23 22:57:41 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-cmullan@ncsu.edu/air.csv


### Initial Data Cleaning

There seem to be a couple issues here: 
1. I'm noticing this data has some -200 values in it, which correspond to missing values - Let's replace those with NULL values. \
I've designed the Proj2Script to properly handle NULL values, so this should ensure our methods work properly.
2. Python isn't happy with the header format. Let's rename some columns for clarity. We can get rid of the first column which is serving as a secondary index.
This is code pulled from our first project, and updated to match the current data processing method.
3. Our "Time" column seems to have assigned today's date to the times. We'll remove that.

In [4]:
airdata.df =  airdata.df.withColumnRenamed("C6H6(GT)", "B")\
                        .withColumnRenamed("PT08.S1(CO)", "CO")\
                        .withColumnRenamed("PT08.S2(NMHC)", "NMHC")\
                        .withColumnRenamed("PT08.S3(NOx)", "NOx")\
                        .withColumnRenamed("PT08.S4(NO2)", "NO2")\
                        .withColumnRenamed("PT08.S5(O3)", "O3")

airdata.df = airdata.df.drop("_c0")

Here, we're replacing those -200 values with NULL.

In [5]:
airdata.df = airdata.df.replace(float(-200), None) ## Replace "-200" with "NaN"

In [6]:
airdata.df.show()

+---------+-------------------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|     Date|               Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|
+---------+-------------------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|3/10/2004|2026-03-23 18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|
|3/10/2004|2026-03-23 19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|
|3/10/2004|2026-03-23 20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|
|3/10/2004|2026-03-23 21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|
|3/10/2004|2026-03-23 22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|
|3/10/2004|2026-03-23 23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|     96|1393| 949|11.2|59.2|0.7848|
|

In [7]:
airdata.df.show()

+---------+-------------------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|     Date|               Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|
+---------+-------------------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|3/10/2004|2026-03-23 18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|
|3/10/2004|2026-03-23 19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|
|3/10/2004|2026-03-23 20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|
|3/10/2004|2026-03-23 21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|
|3/10/2004|2026-03-23 22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|
|3/10/2004|2026-03-23 23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|     96|1393| 949|11.2|59.2|0.7848|
|

Great! Looks like our values have been replaced! To be sure, let's run minmax.

In [8]:
airdata.minmax()

<class 'NoneType'>
Column Provided: False
Grouping Var Provided: False


,0
min(CO(GT)),0.1000
max(CO(GT)),11.9000
min(CO),647.0000
max(CO),2040.0000
min(NMHC(GT)),7.0000
max(NMHC(GT)),1189.0000
min(B),0.1000
max(B),63.7000
min(NMHC),383.0000
max(NMHC),2214.0000


Perfect! No -200 values to be seen!

Now, for the Time column:

In [9]:
airdata.df = airdata.df.withColumn("Time", F.date_format(F.col("Time"), "HH:mm:ss"))

In [10]:
airdata.df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|
|3/10/2004|23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|     96|1393| 949|11.2|59.2|0.7848|
|3/11/2004|00:00:00|   1.2|1185|      31| 3.6| 690|     62|1462|     77|1333| 733|11.3|56.8|0.7603|


Great! Now we have a date, and a time column. \
Finally, I am going to make a list of our column names for use later.

In [11]:
OGcols = airdata.df.columns
OGcols

['Date',
 'Time',
 'CO(GT)',
 'CO',
 'NMHC(GT)',
 'B',
 'NMHC',
 'NOx(GT)',
 'NOx',
 'NO2(GT)',
 'NO2',
 'O3',
 'T',
 'RH',
 'AH']

## Method Testing

Here, I give some examples of the methods I've written, to show that they work.
I will provide 4-5 examples for each.

### Validation Methods

#### withinlimits()

This method takes in an item of SparkDataCheck class, a specific column name from that item's df attribute, and an upper and lower numeric bound. \
Then, a new column is appended providing True or False values for each row in the specified column, depending on if it's in the specified range. \
Only one bound is required, and if only one is provided a one-sided check will be made.
This method returns the edited SparkDataCheck item, so .show() must be used to see the table.

##### Numeric column specified, upper and lower bound supplied:

Here, you can see a new column on the right, corresponding to if each value of T is within the specified 10-11 range. \
This column is added to the SparkDataCheck item in use.

In [12]:
airdata.withinlimits("T", lower = 10, upper = 11).df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|T within (10,11)|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|           false|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|           false|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|           false|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|            true|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|           false|
|3/10/2004|23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|

##### Numeric column specified, only lower bound supplied:

Here, you can see a new column on the right, corresponding to if each value of T is below the given value 1.5. \
This column is added to the SparkDataCheck item in use.

In [13]:
airdata.withinlimits("CO(GT)", lower = 1.5).df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+----------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|T within (10,11)|CO(GT) above 1.5|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+----------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|           false|            true|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|           false|            true|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|           false|            true|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|            true|            true|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|120

##### Non-numeric column specified, lower bound supplied:

When a non-numeric column is specified, the method makes no changes and returns your object unedited. A message is printed, telling you what's wrong.

In [14]:
airdata.withinlimits("T within (10,11)", lower = 50).df.show()

Non-numeric column supplied - No actions taken
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+----------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|T within (10,11)|CO(GT) above 1.5|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+----------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|           false|            true|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|           false|            true|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|           false|            true|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|            true|            true|
|3/10/2004|22:0

##### Numeric column specified, only upper bound given:

This is similar to the earlier example with CO(GT). With only an upper bound given, every value <= 1200 is True, while all others are False.

In [15]:
airdata.withinlimits("CO", upper = 1200).df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+----------------+-------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|T within (10,11)|CO(GT) above 1.5|CO below 1200|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+----------------+-------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|           false|            true|        false|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|           false|            true|        false|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|           false|            true|        false|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|            t

Just for ease of viewing, I will remove these additional columns now.

In [16]:
airdata.df = airdata.df.select(OGcols)
airdata.df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|
|3/10/2004|23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|     96|1393| 949|11.2|59.2|0.7848|
|3/11/2004|00:00:00|   1.2|1185|      31| 3.6| 690|     62|1462|     77|1333| 733|11.3|56.8|0.7603|


#### onlist()

This method takes in an item of SparkDataCheck class, a specific column name from that item's df attribute, and a list of strings. \
Then, a new column is appended providing True or False values for each row in the specified column, depending on if that value is on the list. \
This method returns the edited SparkDataCheck item, so .show() must be used to see the table.

##### String column supplied, string supplied:

Here, you can see a new column on the right, corresponding to if each cell in "Date" is 3/10/2004. \
This column is added to the SparkDataCheck item in use.

In [17]:
airdata.onlist("Date", ["3/10/2004"]).df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|Date on list|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|        true|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|        true|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|        true|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|        true|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|        true|
|3/10/2004|23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|     96|1393| 949|11.2|59.2|0.78

##### String column supplied, list of strings supplied:

Here, you can see a new column on the right, corresponding to if each cell in "Time" is in the list we provided.

In [18]:
airdata.onlist("Time", ["19:00:00", "21:00:00", "09:00:00"]).df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|Date on list|Time on list|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|        true|       false|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|        true|        true|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|        true|       false|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|        true|        true|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|        true|      

##### String column specified, non-numeric list supplied:

In this case, the method will determine that the list contains non-strings, print the issue, and return the unedited object.

In [19]:
airdata.onlist("Time", [18,19,20]).df.show()

Non-string list supplied - No actions taken
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|Date on list|Time on list|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|        true|       false|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|        true|        true|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|        true|       false|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|        true|        true|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|14

##### Non-string column specified, valid list given:

In this case, it will return the unedited object and print the issue.

In [20]:
airdata.onlist("B", ["11.9", "4.7"]).df.show()

Non-string column supplied - No actions taken
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|Date on list|Time on list|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|        true|       false|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|        true|        true|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|        true|       false|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|        true|        true|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|

##### Once again, clearing the new columns

In [21]:
airdata.df = airdata.df.select(OGcols)
airdata.df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|
|3/10/2004|23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|     96|1393| 949|11.2|59.2|0.7848|
|3/11/2004|00:00:00|   1.2|1185|      31| 3.6| 690|     62|1462|     77|1333| 733|11.3|56.8|0.7603|


#### nulltest()

This method takes in an item of SparkDataCheck class and a specific column name from that item's df attribute. \
Then, a new column is appended providing True or False values for each row in the specified column, depending on if that value is NULL. \
This method returns the edited SparkDataCheck item, so .show() must be used to see the table.

In [22]:
airdata.nulltest("NO2(GT)").df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+---------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|NO2(GT) is null|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+---------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|          false|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|          false|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|          false|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|          false|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|          false|
|3/10/2004|23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|     96|

This method can be used pretty quickly to check counts of null values using .groupBy().

In [23]:
airdata.nulltest("NOx(GT)").df.groupBy("Time", "NOx(GT) is null").count().sort("Time").show()

+--------+---------------+-----+
|    Time|NOx(GT) is null|count|
+--------+---------------+-----+
|00:00:00|          false|  336|
|00:00:00|           true|   54|
|01:00:00|           true|   56|
|01:00:00|          false|  334|
|02:00:00|           true|   58|
|02:00:00|          false|  332|
|03:00:00|           true|  364|
|03:00:00|          false|   26|
|04:00:00|           true|   58|
|04:00:00|          false|  332|
|05:00:00|           true|   57|
|05:00:00|          false|  333|
|06:00:00|           true|   56|
|06:00:00|          false|  334|
|07:00:00|           true|   56|
|07:00:00|          false|  334|
|08:00:00|           true|   61|
|08:00:00|          false|  329|
|09:00:00|          false|  334|
|09:00:00|           true|   56|
+--------+---------------+-----+
only showing top 20 rows


In [24]:
airdata.nulltest("CO(GT)").df.groupBy("CO(GT) is null").count().show()

+--------------+-----+
|CO(GT) is null|count|
+--------------+-----+
|          true| 1683|
|         false| 7674|
+--------------+-----+



In [25]:
airdata.nulltest("B").df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+---------------+---------------+--------------+---------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|NO2(GT) is null|NOx(GT) is null|CO(GT) is null|B is null|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+---------------+---------------+--------------+---------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|          false|          false|         false|    false|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|          false|          false|         false|    false|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|          false|          false|         false|    false|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172

### Summarization Methods

The methods below take in a SparkDataCheck object, and analyze it in different ways. Each returns a pandas data frame.

#### minmax()

This method accepts a specific column name, as well as a grouping variable, `groupvar`. It will return a pandas df containing the minimum and maximum values for the specified column within the SparkDataCheck df attribute, grouped by `groupvar` if one is provided. \
If no column is provided, this method will return a df of the min and max values for every numeric column in the object's df attribute, grouped if specified.

In [26]:
airdata.minmax("T")

<class 'str'>
Column Provided: True
Grouping Var Provided: False


,min(T),max(T)
0,-1.9,44.6


As I did in the "Data Cleaning" section above, you can also use `.minmax()` to check for missing values in the data. Because -200 is far below the possible values for these variables, it would appear as the min value for any column with a missing value - using `.minmax()` and seeing no -200 values showed me that our replace method worked.

In [27]:
airdata.minmax()

<class 'NoneType'>
Column Provided: False
Grouping Var Provided: False


,0
min(CO(GT)),0.1000
max(CO(GT)),11.9000
min(CO),647.0000
max(CO),2040.0000
min(NMHC(GT)),7.0000
max(NMHC(GT)),1189.0000
min(B),0.1000
max(B),63.7000
min(NMHC),383.0000
max(NMHC),2214.0000


Here, I am chaining our `minmax()` method with a `.sort_values()` call, which helps sort the data by the column we want.

In [28]:
airdata.minmax("NOx", "Time").sort_values("Time")

<class 'str'>
Column Provided: True
Grouping Var Provided: True


,Time,min(NOx),max(NOx)
8,00:00:00,443,1627
1,01:00:00,435,1900
0,02:00:00,497,1955
13,03:00:00,530,2081
14,04:00:00,508,2331
2,05:00:00,515,2559
4,06:00:00,534,2294
7,07:00:00,407,2683
15,08:00:00,370,2327
5,09:00:00,345,2318


This method can be useful when combined with the validation methods from earlier: \
I can create and use a new category in one line - Here, finding the min and max levels for RH at different B levels.

In [29]:
airdata.withinlimits("B", upper = 5).minmax("RH", groupvar = "B below 5")

<class 'str'>
Column Provided: True
Grouping Var Provided: True


,B below 5,min(RH),max(RH)
0,None,NaN,NaN
1,True,10.0,87.2
2,False,9.2,88.7


#### strcount()

This method accepts a column name, and a second optional column name. It returns a pandas df with the count of occurences for each string that appears in that column. \
If two columns are supplied, it provides a pandas df with the count of each unique combination of the strings in each column.

In this first example, it's no surprise each date appears 24 times, once per hour.

In [30]:
airdata.strcount("Date")

,Date,count
0,9/2/2004,24
1,12/26/2004,24
2,2/18/2005,24
3,10/10/2004,24
4,10/11/2004,24
...,...,...
386,1/23/2005,24
387,6/28/2004,24
388,8/16/2004,24
389,12/20/2004,24


This method checks to make sure each column supplied is comprised of strings before running - If a non-string column is supplied, that information is printed.

In [31]:
airdata.strcount("Date", "CO")

Non-string second column supplied - No actions taken


In [32]:
airdata.strcount("NOx")

Non-string first column supplied - No actions taken


Here, again, no surprise that each combination of dates & times happens once.

In [33]:
airdata.strcount("Date", "Time")

,Date,Time,count
5189,1/1/2005,09:00:00,1
8201,1/1/2005,14:00:00,1
8627,1/1/2005,10:00:00,1
1901,1/1/2005,17:00:00,1
3070,1/1/2005,07:00:00,1
...,...,...,...
6447,9/9/2004,23:00:00,1
625,9/9/2004,13:00:00,1
5128,9/9/2004,10:00:00,1
6582,9/9/2004,02:00:00,1


### Re-reading in the data

In [34]:
pandasairdata = pd.read_csv("air.csv")

In [39]:
pad = SparkDataCheck.frompdf(pandasairdata, spark)
pad.df.show()

+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|       1376|      80|     9.2|        

In [38]:
pad.withinlimits("CO(GT)", lower = 1, upper = 2).df.show()

+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------------------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|CO(GT) within (1,2)|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-------------------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|              false|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|               true|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|      

# Part II

Part II Description coming soon

Importing additional tools

In [2]:
import pyspark.pandas as ps

Let's set our spark session again:

In [19]:
spark = SparkSession.builder.appName('my_app').getOrCreate()

### Pandas on Spark 

In [9]:
#spark.conf.set("spark.sql.ansi.enabled", "false")

And let's read in our csv:

In [89]:
nfl = ps.read_csv("weekly_nfl_data.csv")

Taking a look at our first 5 rows:

In [93]:
nfl[0:5]

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


And our column names:

In [94]:
nfl.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

Per the instructions, here I am shrinking our df by keeping only rows for QBs in the regular season, from 2005-2023. \
Then, I am removing many columns we won't be needing for this project.

In [95]:
## QBs only, regular seasons 2005-2023

qb = nfl[(nfl["position"] == "QB") & (nfl["season_type"] == "REG") & (nfl["season"].between(2005,2023))] 

## Keeping columns we want.

qb = qb.loc[:,["player_display_name", "season", "week", "completions", "attempts", "passing_yards", "passing_tds", "interceptions"]]

Here, I will find the mean and sum for each player's season values, using the `.agg` function. \
I will also save this as a new object, so I can continue to work on it.

In [96]:
qbsummary = qb.groupby(["player_display_name", "season"]).agg(["sum", "mean"]).reset_index()
qbsummary[0:5]

player_display_name season week            completions            attempts            passing_yards             passing_tds           interceptions          
                              sum       mean         sum       mean      sum       mean           sum        mean         sum      mean           sum      mean
0       Jake Delhomme   2006   99   7.615385         263  20.230769      431  33.153846        2805.0  215.769231          17  1.307692          11.0  0.846154
1        Jake Plummer   2005  144   9.000000         277  17.312500      456  28.500000        3366.0  210.375000          18  1.125000           7.0  0.437500
2         Matt Schaub   2006   60  12.000000          18   3.600000       27   5.400000         208.0   41.600000           1  0.200000           2.0  0.400000
3         Vince Young   2006  143   9.533333         184  12.266667      356  23.733333        2199.0  146.600000          12  0.800000          13.0  0.866667
4       Kerry Collins   2007   48   8.000000          50   8.333333       82  13.666667         531.0   88.500000           0  0.000000           0.0  0.000000

Here I'm going to "flatten" my df and combine the two header rows. This will make the next steps easier!

In [97]:
qbsummary.columns = ['_'.join(col) for col in qbsummary.columns]
qbsummary[0:5]

,player_display_name_,season_,week_sum,week_mean,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean
0,Jake Delhomme,2006,99,7.615385,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154
1,Jake Plummer,2005,144,9.000000,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500
2,Matt Schaub,2006,60,12.000000,18,3.600000,27,5.400000,208.0,41.600000,1,0.200000,2.0,0.400000
3,Vince Young,2006,143,9.533333,184,12.266667,356,23.733333,2199.0,146.600000,12,0.800000,13.0,0.866667
4,Kerry Collins,2007,48,8.000000,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,0.0,0.000000


Here I'm adding two new columns, **Completion Percentage** and **TD-Int ratio**

In [98]:
qbsummary["completion_percentage"] = (qbsummary["completions_sum"]/qbsummary["attempts_sum"])
qbsummary["td_int_ratio"] = (qbsummary["passing_tds_sum"]/qbsummary["interceptions_sum"])
qbsummary[0:5]

,player_display_name_,season_,week_sum,week_mean,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
0,Jake Delhomme,2006,99,7.615385,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154,0.610209,1.545455
1,Jake Plummer,2005,144,9.000000,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500,0.607456,2.571429
2,Matt Schaub,2006,60,12.000000,18,3.600000,27,5.400000,208.0,41.600000,1,0.200000,2.0,0.400000,0.666667,0.500000
3,Vince Young,2006,143,9.533333,184,12.266667,356,23.733333,2199.0,146.600000,12,0.800000,13.0,0.866667,0.516854,0.923077
4,Kerry Collins,2007,48,8.000000,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,0.0,0.000000,0.609756,NaN


Now I'm going to subset to only rows with 50+ attempts, and display our top 40 values of **Completion Percentage**, and then **TD-Int Ratio** in the following cell.

In [99]:
qbsummary = qbsummary[qbsummary["attempts_sum"] >= 50]
qbsummary.sort_values("completion_percentage", ascending = False)[0:40]

,player_display_name_,season_,week_sum,week_mean,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
1409,C.J. Beathard,2023,65,10.833333,40,6.666667,53,8.833333,349.0,58.166667,1,0.166667,0.0,0.000000,0.754717,inf
1248,Colt McCoy,2021,62,8.857143,74,10.571429,99,14.142857,740.0,105.714286,3,0.428571,1.0,0.142857,0.747475,3.000000
900,Matt Schaub,2019,52,10.400000,50,10.000000,67,13.400000,580.0,116.000000,3,0.600000,1.0,0.200000,0.746269,3.000000
953,Drew Brees,2018,130,8.666667,364,24.266667,489,32.600000,3992.0,266.133333,32,2.133333,5.0,0.333333,0.744376,6.400000
1057,Drew Brees,2019,119,10.818182,281,25.545455,378,34.363636,2979.0,270.818182,27,2.454545,4.0,0.363636,0.743386,6.750000
1338,Mason Rudolph,2023,66,16.500000,55,13.750000,74,18.500000,719.0,179.750000,3,0.750000,0.0,0.000000,0.743243,inf
1133,Taysom Hill,2020,147,9.187500,88,5.500000,121,7.562500,928.0,58.000000,4,0.250000,2.0,0.125000,0.727273,2.000000
1007,Nick Foles,2018,51,10.200000,141,28.200000,195,39.000000,1413.0,282.600000,7,1.400000,4.0,0.800000,0.723077,1.750000
917,Drew Brees,2017,148,9.250000,386,24.125000,536,33.500000,4334.0,270.875000,23,1.437500,8.0,0.500000,0.720149,2.875000
851,Sam Bradford,2016,146,9.733333,395,26.333333,552,36.800000,3877.0,258.466667,20,1.333333,5.0,0.333333,0.715580,4.000000


Wow! A couple infinite values because they had no interceptions!

In [100]:
qbsummary.sort_values("td_int_ratio", ascending = False)[0:40]

,player_display_name_,season_,week_sum,week_mean,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
6,Charlie Batch,2006,71,10.142857,30,4.285714,52,7.428571,477.0,68.142857,5,0.714286,0.0,0.000000,0.576923,inf
26,Matt Schaub,2005,65,9.285714,33,4.714286,64,9.142857,495.0,70.714286,4,0.571429,0.0,0.000000,0.515625,inf
73,Todd Collins,2007,62,15.500000,67,16.750000,105,26.250000,888.0,222.000000,5,1.250000,0.0,0.000000,0.638095,inf
455,Troy Smith,2007,62,15.500000,40,10.000000,76,19.000000,452.0,113.000000,2,0.500000,0.0,0.000000,0.526316,inf
520,Jake Locker,2011,51,10.200000,34,6.800000,66,13.200000,542.0,108.400000,4,0.800000,0.0,0.000000,0.515152,inf
775,Brian Hoyer,2016,27,4.500000,134,22.333333,200,33.333333,1445.0,240.833333,6,1.000000,0.0,0.000000,0.670000,inf
778,Nick Foles,2016,17,8.500000,36,18.000000,55,27.500000,410.0,205.000000,3,1.500000,0.0,0.000000,0.654545,inf
812,Derek Anderson,2014,30,6.000000,65,13.000000,97,19.400000,701.0,140.200000,5,1.000000,0.0,0.000000,0.670103,inf
940,Jimmy Garoppolo,2016,49,8.166667,43,7.166667,63,10.500000,502.0,83.666667,4,0.666667,0.0,0.000000,0.682540,inf
984,Matt Moore,2019,54,9.000000,59,9.833333,91,15.166667,659.0,109.833333,4,0.666667,0.0,0.000000,0.648352,inf


### Spark SQL DataFrame

Here, I'm doing everything I did above, but in the Spark SQL DataFrame format, as opposed to the pandas-on-spark version above.

Reading in the data, and making sure its the proper type:

In [101]:
sql_nfl = spark.read.load("weekly_nfl_data.csv", format = "csv", sep = ",", inferSchema = "true", header = "true")
type(sql_nfl)

pyspark.sql.classic.dataframe.DataFrame

Taking a look at the first 5 rows - Bit hard to look at, but if you zoom (way) out, it's a normal df.

In [102]:
sql_nfl.show(5)

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sack_

Column names:

In [ ]:
sql_nfl.columns

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

Subsetting by position, year, season type, and choosing the columns we want: \
*so efficient, just one line of code!*

In [112]:
sql_qb = sql_nfl.filter(
    (sql_nfl["position"] == "QB") & 
    (sql_nfl["season_type"] == "REG") & 
    (sql_nfl["season"].between(2005, 2023)))\
    .select(
    "player_display_name", 
    "season", 
    "week", 
    "completions", 
    "attempts", 
    "passing_yards", 
    "passing_tds", 
    "interceptions")

sql_qb.show(5)

+-------------------+------+----+-----------+--------+-------------+-----------+-------------+
|player_display_name|season|week|completions|attempts|passing_yards|passing_tds|interceptions|
+-------------------+------+----+-----------+--------+-------------+-----------+-------------+
|         Tony Banks|  2005|  17|         14|      25|        173.0|          1|          2.0|
|      Charlie Batch|  2005|   9|          9|      16|         65.0|          0|          1.0|
|      Charlie Batch|  2005|  10|         13|      19|        150.0|          0|          0.0|
|      Charlie Batch|  2005|  16|          1|       1|         31.0|          1|          0.0|
|         Jeff Blake|  2005|   2|          1|       1|         11.0|          0|          0.0|
+-------------------+------+----+-----------+--------+-------------+-----------+-------------+
only showing top 5 rows


This step is pretty involved - \

* I am creating a grouped dataframe for each combination of `player_display_name` and `season`, with the sums of the other variables. \
* I am then left-joining that with a second df of the same grouping, but with means instead of sums. \
* Next, I am dropping the sum and average of season columns (We didn't want that one, but it was easier to do it this way) \
* Finally, I am using `.show()` to print the top 5 rows.

In [114]:
sql_qbsummary = sql_qb.groupBy(["player_display_name", "season"]).sum()\
        .join(sql_qb.groupBy(["player_display_name", "season"]).mean(),
              on = ["player_display_name", "season"], how = "left")\
        .drop("sum(season)", "avg(season)")

sql_qbsummary.show(5)

+-------------------+------+---------+----------------+-------------+------------------+----------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+
|player_display_name|season|sum(week)|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|        avg(week)|  avg(completions)|     avg(attempts)|avg(passing_yards)|  avg(passing_tds)|avg(interceptions)|
+-------------------+------+---------+----------------+-------------+------------------+----------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+
|      Jake Delhomme|  2006|       99|             263|          431|            2805.0|              17|              11.0|7.615384615384615| 20.23076923076923| 33.15384615384615|215.76923076923077|1.3076923076923077|0.8461538461538461|
|       Jake Plummer|  2005|      144|          

Next, we are recreating our two new stats - Completion percentage, and TD-Int ratio.\
We will do this using `.withColumn()` to specify our new column info. \
\
Here, note that when dividing by zero for "td_int_ratio", Spark SQL produces a `NULL` value, instead of the `inf` value we got with pandas-on-Spark. 

In [116]:
sql_qbsummary = sql_qbsummary.withColumn("completion_percentage", 
                         (sql_qbsummary["sum(completions)"] / sql_qbsummary["sum(attempts)"]))\
            .withColumn("td_int_ratio", (sql_qbsummary["sum(passing_tds)"]/sql_qbsummary["sum(interceptions)"]))

sql_qbsummary.show(5)

+-------------------+------+---------+----------------+-------------+------------------+----------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+
|player_display_name|season|sum(week)|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|        avg(week)|  avg(completions)|     avg(attempts)|avg(passing_yards)|  avg(passing_tds)|avg(interceptions)|completion_percentage|      td_int_ratio|
+-------------------+------+---------+----------------+-------------+------------------+----------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+
|      Jake Delhomme|  2006|       99|             263|          431|            2805.0|              17|              11.0|7.615384615384615| 20.23076923076923| 3

Next, we are removing any QB instance where they had less than 50 attempts in the season. \
*See ya later 2006 Matt Schaub!*

In [117]:
sql_qbsummary = sql_qbsummary.filter(F.col("sum(attempts)") >= 50)
sql_qbsummary.show()

+-------------------+------+---------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+
|player_display_name|season|sum(week)|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|         avg(week)|  avg(completions)|     avg(attempts)|avg(passing_yards)|  avg(passing_tds)|avg(interceptions)|completion_percentage|      td_int_ratio|
+-------------------+------+---------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+
|      Jake Delhomme|  2006|       99|             263|          431|            2805.0|              17|              11.0| 7.615384615384615| 20.2307692307692

Now, let's take a look at our top 40 seasons by **Completion Percentage**: \
*a redemption for 2019 Matt Schaub*

In [122]:
sql_qbsummary.sort("completion_percentage", ascending = False).show(40)

+-------------------+------+---------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+---------------------+------------------+
|player_display_name|season|sum(week)|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|         avg(week)|  avg(completions)|     avg(attempts)|avg(passing_yards)|   avg(passing_tds)| avg(interceptions)|completion_percentage|      td_int_ratio|
+-------------------+------+---------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+---------------------+------------------+
|      C.J. Beathard|  2023|       65|              40|           53|             349.0|               1|               0.0|10.833333333333334| 6.66666666

Same thing here but with TD_int_ratio. \
Note that our `NULL` values aren't considered "higher" than our regular numeric values, and thus are omitted from the top of the list. \
Using a different approach to sorting, we can make sure those QBs who didn't throw an interception *all season* are rewarded for their achievement

In [131]:
sql_qbsummary.sort("td_int_ratio", ascending = False).show(5)

+-------------------+------+---------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+---------------------+------------+
|player_display_name|season|sum(week)|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|         avg(week)|  avg(completions)|     avg(attempts)|avg(passing_yards)|  avg(passing_tds)| avg(interceptions)|completion_percentage|td_int_ratio|
+-------------------+------+---------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+---------------------+------------+
|          Tom Brady|  2016|      134|             291|          432|            3554.0|              28|               2.0|11.166666666666666|             24.25|             

`.desc_nulls_first()` specifies a descending list, with `NULL` values listed first.

In [130]:
sql_qbsummary.sort(F.col("td_int_ratio").desc_nulls_first()).show(40)

+-------------------+------+---------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+---------------------+-----------------+
|player_display_name|season|sum(week)|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|         avg(week)|  avg(completions)|     avg(attempts)|avg(passing_yards)|   avg(passing_tds)| avg(interceptions)|completion_percentage|     td_int_ratio|
+-------------------+------+---------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+---------------------+-----------------+
|      Kerry Collins|  2007|       48|              50|           82|             531.0|               0|               0.0|               8.0| 8.33333333333